# Benchmark: ARIMA Statistical Baseline

**Author** : Anwesha Singh — CSE, Manipal University Jaipur

---

## Purpose

ARIMA serves as the canonical **univariate statistical baseline**.  
Because ARIMA cannot ingest multi-dimensional feature matrices, it operates  
exclusively on the **target time series** `energy_delta_k`.  
This structural limitation must be clearly stated in the paper:

> *"The ARIMA model was evaluated in a univariate setting owing to its  
> architectural constraint of accepting a single input series.  
> All other models were trained on the full multi-variate feature set."*

This distinction does **not** disadvantage the comparison — reviewers  
recognise ARIMA as a well-understood statistical floor, not a deep learning  
competitor.

### What this notebook does
1. Loads the frozen canonical splits for a given horizon.  
2. Reconstructs the raw `y` series (train → val → test in strict order).  
3. Fits an ARIMA model by grid-searching over `(p, d, q)` using AIC.  
4. Generates a rolling one-step-ahead forecast over the test window.  
5. Saves predictions, metrics, and diagnostic plots.


In [ ]:
import os, sys, json, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# statsmodels provides the ARIMA implementation used throughout
try:
    from statsmodels.tsa.arima.model import ARIMA
    from statsmodels.tsa.stattools  import adfuller
except ImportError:
    raise ImportError(
        "statsmodels is not installed.  "
        "Run:  pip install statsmodels"
    )

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
sys.path.insert(0, PROJECT_ROOT)

from core.config  import SPLITS_DIR, BENCHMARK_DIR, FORECAST_HORIZONS
from benchmarking.utils.data_loader import load_splits
from benchmarking.utils.metrics     import compute_metrics, save_benchmark_results

sns.set(style="whitegrid")
print("Environment ready.")

In [ ]:
# ============================================================
#  SET HORIZON before running.  Must match an existing split.
# ============================================================
HORIZON = 60

assert HORIZON in FORECAST_HORIZONS, f"Invalid horizon {HORIZON}"

X_train, y_train, X_val, y_val, X_test, y_test, scaler, meta = \
    load_splits(SPLITS_DIR, HORIZON)

print(f"Horizon k = {HORIZON} s")
print(f"Train sequences : {len(y_train):,}")
print(f"Val   sequences : {len(y_val):,}")
print(f"Test  sequences : {len(y_test):,}")

---
## 1 · Stationarity Check

The Augmented Dickey-Fuller test confirms whether the target series  
is stationary.  ARIMA differencing order `d` is selected accordingly.

In [ ]:
# Use a representative subset for the ADF test to reduce computation time
MAX_ADF = 20_000
sample  = y_train[:MAX_ADF]

adf_stat, p_val, _, _, crit_vals, _ = adfuller(sample, autolag="AIC")

print("Augmented Dickey-Fuller Test on y_train (first {:,} samples)".format(len(sample)))
print(f"  ADF statistic : {adf_stat:.4f}")
print(f"  p-value       : {p_val:.6f}")
print(f"  Critical values:")
for lv, cv in crit_vals.items():
    print(f"    {lv} : {cv:.4f}")

is_stationary = p_val < 0.05
print(f"\n  Series is {'stationary' if is_stationary else 'non-stationary'} at α = 0.05.")
DIFF_ORDER = 0 if is_stationary else 1
print(f"  Recommended differencing order d = {DIFF_ORDER}")

---
## 2 · (p, q) Grid Search via AIC

A small grid is searched to keep runtime tractable.  
A training subset is used; ARIMA scales poorly to millions of rows.

In [ ]:
# ARIMA training subset — use first N samples from the combined train+val series
# to keep fitting time manageable while still capturing trend structure.
MAX_ARIMA_TRAIN = 50_000

# Concatenate train and val targets in chronological order
y_trainval = np.concatenate([y_train, y_val])
y_fit      = y_trainval[:MAX_ARIMA_TRAIN]

# Grid over candidate AR and MA orders
P_RANGE = range(0, 4)   # AR order
Q_RANGE = range(0, 4)   # MA order
D       = DIFF_ORDER

best_aic    = np.inf
best_order  = (1, D, 1)
aic_results = []

print(f"Grid searching ARIMA(p, {D}, q) — training on first {len(y_fit):,} samples ...")

for p in P_RANGE:
    for q in Q_RANGE:
        try:
            mdl = ARIMA(y_fit, order=(p, D, q)).fit()
            aic_results.append({"p": p, "d": D, "q": q, "AIC": mdl.aic})
            if mdl.aic < best_aic:
                best_aic   = mdl.aic
                best_order = (p, D, q)
        except Exception:
            pass

aic_df = pd.DataFrame(aic_results).sort_values("AIC").reset_index(drop=True)
print("\nTop 5 configurations by AIC:")
print(aic_df.head(5).to_string(index=False))
print(f"\nBest order selected : ARIMA{best_order}")

---
## 3 · Rolling One-Step-Ahead Forecast

The model is re-fitted on an expanding window as each test observation  
is consumed.  This is the most defensible evaluation strategy for ARIMA  
in a research context, as it avoids multi-step forecast accumulation error.

In [ ]:
# Rolling forecast — limit test size for feasibility
MAX_TEST = min(len(y_test), 5_000)
y_test_subset = y_test[:MAX_TEST]

# Initialise history with the full fitted training window
history = list(y_fit)
y_pred_arima = []

print(f"Rolling forecast over {MAX_TEST:,} test samples ...")

for t, obs in enumerate(y_test_subset):
    try:
        fit = ARIMA(history, order=best_order).fit()
        fc  = float(fit.forecast(steps=1)[0])
    except Exception:
        fc = float(np.mean(history[-100:]))

    y_pred_arima.append(fc)
    history.append(obs)   # expand window with the true observed value

    if (t + 1) % 500 == 0:
        print(f"  ... {t + 1}/{MAX_TEST} done")

y_pred_arima = np.array(y_pred_arima, dtype=np.float32)
print("Rolling forecast complete.")

---
## 4 · Evaluation and Artefact Saving

In [ ]:
metrics_arima = save_benchmark_results(
    model_name     = "arima",
    horizon        = HORIZON,
    y_true         = y_test_subset,
    y_pred         = y_pred_arima,
    benchmark_root = BENCHMARK_DIR,
    extra_meta     = {
        "order"          : best_order,
        "best_aic"       : float(best_aic),
        "training_size"  : int(MAX_ARIMA_TRAIN),
        "test_subset"    : int(MAX_TEST),
        "note"           : "Univariate model — uses target series only",
    },
)

print(f"ARIMA{best_order}  →  {metrics_arima}")

---
## 5 · Diagnostic Plots

In [ ]:
arima_plot_dir = os.path.join(BENCHMARK_DIR, f"horizon_{HORIZON}", "arima", "plots")
os.makedirs(arima_plot_dir, exist_ok=True)

n_show = min(300, len(y_pred_arima))

# ── Time-series overlay ─────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(y_test_subset[:n_show], label="Actual",    linewidth=1.2)
ax.plot(y_pred_arima[:n_show],  label="ARIMA",     linewidth=1.2, linestyle="--")
ax.set_xlabel("Test sample index")
ax.set_ylabel(f"ΔE$_{{k={HORIZON}}}$ (kWh)")
ax.set_title(f"ARIMA{best_order} — Actual vs Predicted  (k = {HORIZON} s)")
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(arima_plot_dir, "arima_overlay.png"), dpi=200)
plt.show(); plt.close()

# ── Residual distribution ───────────────────────────────────────────────────
residuals = y_test_subset - y_pred_arima
fig, ax = plt.subplots(figsize=(7, 4))
sns.histplot(residuals, bins=50, kde=True, ax=ax, color="steelblue")
ax.axvline(0, color="red", linestyle="--", linewidth=1.2)
ax.set_xlabel("Residual (kWh)")
ax.set_title(f"ARIMA Residual Distribution  (k = {HORIZON} s)")
plt.tight_layout()
plt.savefig(os.path.join(arima_plot_dir, "arima_residuals.png"), dpi=200)
plt.show(); plt.close()

# ── AIC surface heatmap ─────────────────────────────────────────────────────
if len(aic_df) >= 4:
    pivot_aic = aic_df.pivot(index="p", columns="q", values="AIC")
    fig, ax   = plt.subplots(figsize=(6, 5))
    sns.heatmap(pivot_aic, annot=True, fmt=".1f", cmap="YlGnBu", ax=ax)
    ax.set_title(f"AIC Surface — ARIMA(p, {D}, q)  (k = {HORIZON} s)")
    plt.tight_layout()
    plt.savefig(os.path.join(arima_plot_dir, "aic_heatmap.png"), dpi=200)
    plt.show(); plt.close()

print(f"ARIMA diagnostic plots saved to : {arima_plot_dir}")